<a href="https://colab.research.google.com/github/AdiY2j/CS6910_Assignment3/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
from torch.autograd import Variable
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
SOS_char = 0
EOS_char = 1
PAD_char = 2
Unknown_char = 3

In [ ]:
class Lang:
    def __init__(self, name):
        self.word2count = {'<' : 1, '>' : 1, '_' : 1, '?' : 1}
        self.word2index = {'<' : 0, '>' : 1, '_' : 2, '?' : 3}
        self.name = name
        self.index2word = {SOS_char : '<', EOS_char : '>', PAD_char : '_', Unknown_char : '?'}
        self.n_chars = 4

    def add_word(self, word):
        for c in word:
            self.add_char(c)

    def add_char(self, char):
        if char not in self.word2index: # If char not present add it in word2index and inc counter
            self.word2index[char] = self.n_chars
            self.word2count[char] = 1
            self.index2word[self.n_chars] = char
            self.n_chars += 1
        else:
            self.word2count[char] += 1 #If char already present just increment counter

In [ ]:
train_data = pd.read_csv('/content/drive/MyDrive/aksharantar_sampled/hin/hin_train.csv')
valid_data = pd.read_csv('/content/drive/MyDrive/aksharantar_sampled/hin/hin_valid.csv')
test_data = pd.read_csv('/content/drive/MyDrive/aksharantar_sampled/hin/hin_test.csv')

train_data = np.array(train_data)
valid_data = np.array(valid_data)
test_data = np.array(test_data)


train_ip_maxlen = max(len(ip) for ip in train_data[ : , 0]) + 2
train_op_maxlen = max(len(op) for op in train_data[ : , 1]) + 2
val_ip_maxlen = max(len(ip) for ip in valid_data[ : , 0]) + 2
val_op_maxlen = max(len(op) for op in valid_data[ : , 1]) + 2
test_ip_maxlen = max(len(ip) for ip in test_data[ : , 0]) + 2
test_op_maxlen = max(len(op) for op in test_data[ : , 1]) + 2

In [ ]:
train_ip_maxlen, train_op_maxlen, val_ip_maxlen, val_op_maxlen, test_ip_maxlen, test_op_maxlen

(26, 22, 24, 22, 28, 22)

In [ ]:
train_X, train_y = train_data[:,0], train_data[:,1]

In [ ]:
valid_X, valid_y = valid_data[:,0], valid_data[:,1]

In [ ]:
train_X

array(['bindhya', 'kirankant', 'yagyopaveet', ..., 'asahmaton',
       'sulgaayin', 'anchuthengu'], dtype=object)

In [ ]:
input_lang, output_lang = Lang('eng'), Lang('hin')
for word in train_X:
  input_lang.add_word(word)
for word in train_y:
  output_lang.add_word(word)

pairs = [[train_X[i], train_y[i]] for i in range(len(train_X))]

In [ ]:
input_lang.n_chars, output_lang.n_chars

(30, 68)

In [ ]:
print(pairs[0])
print(input_lang.n_chars, output_lang.n_chars)

['bindhya', 'बिन्द्या']
30 68


In [ ]:
len(output_lang.word2count)

68

In [ ]:
input_word, output_word = pairs[0][0], pairs[0][1]
encoded_output = [output_lang.word2index[char] for char in output_word]
print(encoded_output)
decoded_output = [output_lang.index2word[i] for i in encoded_output]
print(decoded_output)

[4, 5, 6, 7, 8, 7, 9, 10]
['ब', 'ि', 'न', '्', 'द', '्', 'य', 'ा']


In [ ]:
def getIndex(lang, word, maxlen):
  index = [SOS_char]
  for i in range(len(word)):
    if word[i] in lang.word2index.keys():
        index.append(lang.word2index[word[i]])
    else:
        index.append(Unknown_char)

  return index

def getWordTensor(lang, word, maxlen):
  index = getIndex(lang, word, maxlen)
  index.append(EOS_char)
  diff = (maxlen - len(index))
  pad_list = [PAD_char] * diff
  index.extend(pad_list)

  return torch.LongTensor(index).to(device)

def getTensorPairs(data, maxlen):
  inputWordTensor = getWordTensor(input_lang, data[0], maxlen)
  outputWordTensor = getWordTensor(output_lang, data[1], maxlen)
  return (inputWordTensor, outputWordTensor)

In [ ]:
class Encoder(nn.Module):
  def __init__(self, input_size, hidden_size, embedding_size, num_layers, dropout_val, cell_type, batch_size, bidirectional=False):
    super(Encoder, self).__init__()
    self.hidden_size = hidden_size
    self.embedding_size = embedding_size
    self.num_layers = num_layers
    self.batch_size = batch_size
    self.cell_type = cell_type
    self.rnn = None
    self.embedding = nn.Embedding(input_size, self.embedding_size)
    self.dropout = nn.Dropout(dropout_val)
    self.bidirectional = bidirectional

    match cell_type:
      case "RNN":
        self.rnn = nn.RNN(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)
      case "LSTM":
        self.rnn = nn.LSTM(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)
      case "GRU":
        self.rnn = nn.GRU(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)

  def initializeHidden(self, batch_size, num_layers):
    num_dir = 1
    if self.bidirectional :
        num_dir = 2
    return torch.zeros(num_layers * num_dir, batch_size, self.hidden_size, device=device)

  def forward(self, input, batch_size, hidden):
    embedded = self.embedding(input).view(1, batch_size, -1)
#     if self.cell_type == "LSTM":
#       output, (hidden, cell) = self.rnn(self.dropout(embedded), (hidden, cell))
#     else:
    output, hidden = self.rnn(self.dropout(embedded), hidden)
    return output, hidden

In [ ]:
class Decoder(nn.Module):
  def __init__(self, hidden_size, output_size, embedding_size, num_layers, dropout_val, cell_type, batch_size, bidirectional):
    super(Decoder, self).__init__()
    self.hidden_size = hidden_size
    self.output_size = output_size
    self.num_layers = num_layers
    self.batch_size = batch_size
    self.cell_type = cell_type
    self.dropout = nn.Dropout(dropout_val)
    self.embedding_size = embedding_size
    self.bidirectional = bidirectional
    self.rnn = None
    self.embedding = nn.Embedding(output_size, self.embedding_size)
    self.num_dir = 1

    match cell_type:
      case "RNN":
        self.rnn = nn.RNN(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)
      case "LSTM":
        self.rnn = nn.LSTM(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)
      case "GRU":
        self.rnn = nn.GRU(self.embedding_size, self.hidden_size, num_layers=self.num_layers, dropout=dropout_val, bidirectional=self.bidirectional)

    if self.bidirectional :
        self.num_dir = 2

    self.out = nn.Linear(self.hidden_size * self.num_dir, output_size)
    self.softmax = nn.LogSoftmax(dim = 1)

#   def initializeHidden(self):
#     return torch.zeros(self.num_layers, self.batch_size, self.hidden_size, device=device)

  def forward(self, input, batch_size, hidden):
    embedded = self.embedding(input).view(1, batch_size, -1)
    embedded = F.relu(self.dropout(embedded))
#     if self.cell_type == "LSTM":
#       output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
#     else :
    output, hidden = self.rnn(embedded, hidden)
    output = self.softmax(self.out(output[0]))
    return output, hidden

In [ ]:
def train_step(batch_size, num_layers, encoder, decoder, inputTensor, outputTensor, enc_optimizer, dec_optimizer, criterion, max_len, teacher_forcing = 0.5):
  loss = 0
  acc = 0
  inputTensor = inputTensor.transpose(0, 1)
  outputTensor = outputTensor.transpose(0, 1)
  ip_len = inputTensor.size(0)
  op_len = outputTensor.size(0)
  enc_hidden = encoder.initializeHidden(batch_size, num_layers)



#   if encoder.bidirectional:
#     enc_outputs = torch.zeros(max_len, encoder.hidden_size*2, device=device)
#   else:
#     enc_outputs = torch.zeros(max_len, encoder.hidden_size, device=device)
#   enc_cell = encoder.initializeHidden(batch_size, num_layers)

  if encoder.cell_type == "LSTM":
    enc_hidden = (enc_hidden, encoder.initializeHidden(batch_size, num_layers))

  enc_optimizer.zero_grad()
  dec_optimizer.zero_grad()

  for i in range(ip_len):
    enc_op, enc_hidden = encoder(inputTensor[i], batch_size, enc_hidden)
#     if encoder.bidirectional:
#         enc_outputs[i] = torch.cat((enc_op[:, :, :encoder.hidden_size], enc_op[:, :, encoder.hidden_size:]), dim=2)
#     else:
#         enc_outputs[i] = enc_op[0, 0]

  dec_hidden = enc_hidden
#   dec_cell = enc_cell


  dec_input = torch.LongTensor([SOS_char]*encoder.batch_size).to(device)
  pred_op = []
  use_teacher_forcing = False

  if random.random() < teacher_forcing:
    use_teacher_forcing = True

  if use_teacher_forcing :
    for i in range(op_len):
      dec_op, dec_hidden = decoder(dec_input, batch_size, dec_hidden)
      loss += criterion(dec_op, outputTensor[i])
      dec_input = outputTensor[i]
#       if dec_input.item() == EOS_char :
#         break
#       else :
#         if dec_input.item() != SOS_char:
#           pred_op += output_lang.index2word[dec_input.item()]

  else :

    for i in range(op_len):
      dec_op, dec_hidden = decoder(dec_input, batch_size, dec_hidden)
      loss += criterion(dec_op, outputTensor[i])
      _, top_i = dec_op.data.topk(1)
#       if top_i.item() == EOS_char :
#         break
#       else:
#         if top_i.item() != SOS_char:
#           pred_op += output_lang.index2word[top_i.item()]

      dec_input = top_i


  loss.backward()

  enc_optimizer.step()
  dec_optimizer.step()

  return loss.item() / op_len

In [ ]:
max_all_len = max(train_ip_maxlen, train_op_maxlen, val_op_maxlen, val_ip_maxlen, test_ip_maxlen, test_op_maxlen)

In [ ]:
max_all_len

28

In [ ]:
train_tensor_pairs = []
for data in pairs:
  train_tensor_pairs.append(getTensorPairs(data, max_all_len))

In [ ]:
valid_pairs = [[valid_X[i], valid_y[i]] for i in range(len(valid_X))]

In [ ]:
valid_tensor_pairs = []
for data in valid_pairs:
    valid_tensor_pairs.append(getTensorPairs(data, max_all_len))

In [ ]:
pairs[0], train_tensor_pairs[0]

(['bindhya', 'बिन्द्या'],
 (tensor([ 0,  4,  5,  6,  7,  8,  9, 10,  1,  2,  2,  2,  2,  2,  2,  2,  2,  2,
           2,  2,  2,  2,  2,  2,  2,  2,  2,  2]),
  tensor([ 0,  4,  5,  6,  7,  8,  7,  9, 10,  1,  2,  2,  2,  2,  2,  2,  2,  2,
           2,  2,  2,  2,  2,  2,  2,  2,  2,  2])))

In [ ]:
valid_pairs[0], valid_tensor_pairs[0]

(['bajai', 'बजाई'],
 (tensor([ 0,  4, 10, 25, 10,  5,  1,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
           2,  2,  2,  2,  2,  2,  2,  2,  2,  2]),
  tensor([ 0,  4, 16, 10, 45,  1,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
           2,  2,  2,  2,  2,  2,  2,  2,  2,  2])))

In [ ]:
def evaluate(batch_size, num_layers, encoder, decoder, loader, input_lang, output_lang, max_len):
    with torch.no_grad():
        correct_ans = 0
        total_samples = 0

        for inputX, labelY in tqdm(loader):
#             enc_cell = encoder.initializeHidden(batch_size, num_layers)
            enc_hidden = encoder.initializeHidden(batch_size, num_layers)

            trans_inp = inputX.transpose(0, 1)
            trans_op = labelY.transpose(0, 1)

            if encoder.cell_type == "LSTM":
                enc_hidden = (enc_hidden, encoder.initializeHidden(batch_size, num_layers))

            ip_len = trans_inp.size(0)
            op_len = trans_op.size(0)

            output = Variable(torch.LongTensor(op_len, batch_size)).to(device)

            if encoder.bidirectional:
                enc_outputs = torch.zeros(max_len, batch_size, encoder.hidden_size*2, device=device)
            else:
                enc_outputs = torch.zeros(max_len, batch_size, encoder.hidden_size, device=device)

            for i in range(ip_len):
                enc_op, enc_hidden = encoder(trans_inp[i], batch_size, enc_hidden)
#                 enc_outputs[i] = enc_op[0]
#                 if encoder.bidirectional:
#                     enc_outputs[i] = torch.cat((enc_op[:, :, :encoder.hidden_size], enc_op[:, :, encoder.hidden_size:]), dim=2)
#                 else:
#                     enc_outputs[i] = enc_op[0, 0]

            dec_hidden = enc_hidden
#             dec_cell = enc_hidden

            dec_input = torch.LongTensor(([SOS_char]*batch_size)).to(device)

            for i in range(op_len):
                dec_op, dec_hidden = decoder(dec_input, batch_size, dec_hidden)
                _, top_i = dec_op.data.topk(1)
                decoder_input = top_i
                output[i] = torch.cat(tuple(top_i))

            output = output.transpose(0, 1)

            output_len = output.size(0)

            for i in range(output_len):
                pred = [output_lang.index2word[c.item()] for c in output[i] if c not in [EOS_char, SOS_char, Unknown_char, PAD_char]]
                target = [output_lang.index2word[c.item()] for c in labelY[i] if c not in [EOS_char, SOS_char, Unknown_char, PAD_char]]
                total_samples += 1
#                 if i < 100:
#                     print(pred, target)
                if pred == target :
                    print(pred, target)
                    correct_ans += 1


    print(correct_ans, total_samples)
    return (correct_ans/total_samples)

In [ ]:
def train(batch_size, num_layers, encoder, decoder, learning_rate, epochs, train_loader, val_loader, input_lang, output_lang, max_len):
    for epoch in range(epochs):
        print('Epoch : {}'.format(epoch+1))
        total_train_loss = 0
        total_val_loss = 0
        train_epoch_loss = 0
        val_epoch_loss = 0
        train_epoch_acc = 0
        correct_ans = 0
        val_correct_ans = 0
        val_acc = 0
        enc_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
        dec_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
        loss_func = nn.NLLLoss()
        train_samples = 0
        val_samples = 0

        for inputWordTensor, outputWordTensor in tqdm(train_loader):
            loss = train_step(batch_size, num_layers, encoder, decoder, inputWordTensor, outputWordTensor, enc_optimizer, dec_optimizer, loss_func, max_len)
            total_train_loss += loss
            train_epoch_loss += loss
            train_samples += 1
#             if i % 1000 == 0:
#                 train_loss_avg = total_train_loss / 1000
#                 total_train_loss = 0
#                 print('Iteration : {}, Train Loss : {:.4f}, Correct Ans : {}'.format(i, train_loss_avg, correct_ans))

        train_epoch_loss = train_epoch_loss / train_samples

        for inputWordTensor, outputWordTensor in tqdm(val_loader):
            val_loss = train_step(batch_size, num_layers, encoder, decoder, inputWordTensor, outputWordTensor, enc_optimizer, dec_optimizer, loss_func, max_len)
            val_epoch_loss += val_loss
            total_val_loss += val_loss
            val_samples += 1

#             if i % 1000 == 0:
#                 val_loss_avg = total_val_loss / 1000
#                 total_val_loss = 0
#                 print('Iteration : {}, Val Loss : {:.4f}, Correct Ans : {}'.format(i, val_loss_avg, val_correct_ans))

        val_epoch_loss = val_epoch_loss / val_samples


        train_epoch_acc = evaluate(batch_size, num_layers, encoder, decoder, train_loader, input_lang, output_lang, max_len)
        val_epoch_acc = evaluate(batch_size, num_layers, encoder, decoder, val_loader, input_lang, output_lang, max_len)

        print('Train Loss : {:.4f}, Val Loss : {:.4f}, Train Acc : {:.4f}, Val Acc : {:.4f}'.format(train_epoch_loss, val_epoch_loss, train_epoch_acc, val_epoch_acc))

In [ ]:
hidden_size = 256
batch_size = 32
embedding_size = 128
dropout = 0
num_layers = 2
cell_type = "LSTM"

train_loader = DataLoader(train_tensor_pairs, batch_size = batch_size, drop_last=True)
val_loader = DataLoader(valid_tensor_pairs, batch_size = batch_size, drop_last=True)

encoder = Encoder(input_lang.n_chars, hidden_size, embedding_size, num_layers, dropout, cell_type, batch_size, bidirectional=True).to(device)
decoder = Decoder(hidden_size, output_lang.n_chars, embedding_size, num_layers, dropout, cell_type, batch_size, bidirectional=True).to(device)
train(batch_size, num_layers, encoder, decoder, 0.01, 10, train_loader, val_loader, input_lang, output_lang, max_all_len)